# Vesuvius Autoresearch: Night Shift Analysis

This notebook provides a comprehensive analysis of the autonomous architectural evolution performed during the Night Shift sprints. It tracks metric improvements, visualizes predictions, and summarizes key insights.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
from PIL import Image
import datetime

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

def load_results(path='results.tsv'):
    if not os.path.exists(path):
        return pd.DataFrame()
    df = pd.read_csv(path, sep='\t')
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df.sort_values('timestamp')

df = load_results()
print(f"Loaded {len(df)} successful experiments.")
df.tail()

## 1. Metric Progress (Cross-Fragment Generalization)

We track the `val_bpb` which, in our current configuration, represents **1.0 - Validation Dice Score**. Lower is better.

In [ ]:
if not df.empty:
    fig, ax1 = plt.subplots(figsize=(14, 7))

    # Primary Metric: Val Dice Loss
    ax1.plot(df['timestamp'], df['val_bpb'], marker='o', color='#2c3e50', linewidth=2, label='1 - Dice (Validation)')
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Dice Loss (Lower is Better)')
    ax1.set_yscale('log')
    
    # Secondary Metric: Throughput
    ax2 = ax1.twinx()
    ax2.plot(df['timestamp'], df['throughput_Mvps'], marker='x', color='#e74c3c', linestyle='--', alpha=0.5, label='Throughput (Mvps)')
    ax2.set_ylabel('Throughput (Mvps)', color='#e74c3c')
    
    plt.title('Autonomous Optimization: Cross-Fragment Generalization Progress')
    fig.tight_layout()
    plt.show()
else:
    print("No data to plot yet. Run a Night Shift sprint first!")

## 2. Visual Validation (Latest Predictions)

Let's look at the actual ink maps generated by the latest models.

In [ ]:
pred_images = sorted(glob.glob('predictions/*.png'))
if pred_images:
    latest_pred = pred_images[-1]
    print(f"Displaying latest prediction: {latest_pred}")
    img = Image.open(latest_pred)
    plt.figure(figsize=(15, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("No prediction images found in predictions/")

## 3. Top Performing Architectures

Summary of the best models discovered so far.

In [ ]:
if not df.empty:
    top_3 = df.nsmallest(3, 'val_bpb')
    print("Top 3 Discovery Milestones:")
    display(top_3[['timestamp', 'val_bpb', 'throughput_Mvps', 'num_params_M']])
    
    baseline = df.iloc[0]['val_bpb']
    best = df['val_bpb'].min()
    improvement = (baseline - best) / baseline * 100
    print(f"\nOverall Dice Loss Improvement: {improvement:.2f}%")